**Repository blueprint:** this Git copy contains explanations
and plotting code, but no patient waveform payload or figure outputs. Generate the self-contained
local notebook and offline HTML with:

```bash
OMP_NUM_THREADS=2 OPENBLAS_NUM_THREADS=2 .venv/bin/python -m scripts.reports.build_real_ecg_notebook
```

Use the resulting `outputs/real_ecg_notebook_2026_10_02/real-ecg-localization.ipynb`
for the portable, runnable version. Raw recordings remain local according to the project's data policy.


# Understanding ECG localization with real recordings

This notebook explains the existing research in plain language. It includes ECGs labeled normal
and ECGs with different recorded abnormalities, then shows two improvements tested against expert
annotations. A normal ECG does not prove that a person has no heart disease.

The portable local version contains its selected real waveforms, saved model outputs, and source
notes. **Restart the kernel and run all cells:** it needs Python, NumPy, Matplotlib, and IPython,
but no repository checkout, model weights, GPU, internet connection, or original data folders.
The HTML export can be read without running Python.

This is a replay and explanation of completed research, not a diagnostic tool. The examples were
chosen for illustration from already-read development data. They do not form an independent test.


## 1. Reading the picture

An ECG records voltage over time. Each lead is a different electrical view of the same heartbeat,
like cameras looking at one event from different angles. These views are not pictures of twelve
separate parts of the heart.

The small P wave reflects electrical activation of the upper chambers. The sharper QRS complex
reflects activation of the lower chambers. The ST segment and T wave describe part of their
electrical recovery. The horizontal axis is time; the vertical axis is recorded voltage.

A wider spike, a premature beat, or a changed recovery shape may deserve attention. Appearance
depends on the lead, and a pattern needs clinical context. Here we retain the database's original
labels instead of inventing diagnoses from the pictures.

Each plot adapts its vertical axis to make the recorded shape readable. Compare the voltage
numbers, rather than the drawn height, across leads and examples. The closer views magnify
the original samples; they do not modify the ECG or add new algorithm predictions.


In [ ]:
import base64
import gzip
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

plt.rcParams.update({"font.size": 10, "figure.dpi": 110})


In [ ]:
raise RuntimeError("This Git copy omits patient waveforms. Generate the portable local "
                   "version with: python -m scripts.reports.build_real_ecg_notebook")

In [ ]:
def plot_record(case, marks=None, limits=None, title=None):
    """Show original recorded voltages and separately stored highlights."""
    signal = np.asarray(case["signal"], dtype=float)
    time = np.asarray(case["time"], dtype=float)
    count = len(case["leads"])
    columns = 2 if count > 2 else 1
    rows = (count + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(15, max(3.5, rows * 1.45)),
                             squeeze=False, sharex=True)
    shown = case.get("marks", []) if marks is None else marks
    for index, axis in enumerate(axes.flat):
        if index >= count:
            axis.set_visible(False)
            continue
        axis.plot(time, signal[index], color="#222222", linewidth=0.65)
        axis.set_ylabel(case["leads"][index] + " / " + case.get("units", "mV"))
        axis.grid(alpha=0.2)
        for mark in shown:
            if mark.get("lead") in (None, -1, index):
                axis.axvspan(mark["start"], mark["end"], color=mark["color"],
                             alpha=0.23, label=mark["label"])
        if limits is not None:
            axis.set_xlim(*limits)
        handles, labels = axis.get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        if unique:
            axis.legend(unique.values(), unique.keys(), fontsize=7, loc="upper right")
        axis.set_xlabel("Time in the original recording (seconds)")
    fig.suptitle(title or case["title"], fontsize=13)
    fig.tight_layout()
    plt.show()
    plt.close(fig)


def plot_zoom(case):
    """Magnify recorded samples without adding or changing localization marks."""
    marks = case.get("marks", [])
    marked = [m['lead'] for m in marks if m.get('lead') is not None]
    leads = list(dict.fromkeys(marked))[:2]
    if not leads:
        leads = [case['leads'].index(name) for name in ['II', 'V5'] if name in case['leads']]
    center = (marks[0]['start'] + marks[0]['end']) / 2 if marks else case['time'][0] + 2
    start = max(case['time'][0], min(center - 1, case['time'][-1] - 2))
    end = start + 2
    signal = np.asarray(case['signal'])
    time = np.asarray(case['time'])
    selected = (time >= start) & (time <= end)
    fig, axes = plt.subplots(len(leads), 1, figsize=(14, 2.2 * len(leads)), squeeze=False)
    for lead, axis in zip(leads, axes.flat):
        axis.plot(time[selected], signal[lead, selected], color='#222222', linewidth=0.9)
        for mark in marks:
            if mark.get('lead') in (None, lead):
                axis.axvspan(mark['start'], mark['end'], color=mark['color'], alpha=0.23,
                             label=mark['label'])
        if axis.get_legend_handles_labels()[0]:
            axis.legend(fontsize=8)
        axis.set_xlim(start, end)
        axis.set_ylabel(case['leads'][lead] + ' / ' + case.get('units', 'mV'))
        axis.set_xlabel('Time in the original recording (seconds)')
        axis.grid(alpha=0.2)
    fig.suptitle('Closer view of the same real samples — ' + case['title'])
    fig.tight_layout()
    plt.show()
    plt.close(fig)


def describe_record(case):
    """Display the saved labels, decisions, and interpretation limits."""
    notes = case.get("annotation_notes", "")
    if isinstance(notes, list):
        notes = "\n\n".join(notes)
    lines = [case.get("plain_language", "")]
    if case.get("codes"):
        lines.append("Recorded annotation codes: `" + str(case["codes"]) + "`.")
    if case.get("annotations"):
        labels = ["| Code | Recorded ECG annotation |", "|---|---|"]
        labels.extend("| " + row["code"] + " | " + row["description"] + " |"
                      for row in case["annotations"])
        lines.append("\n".join(labels))
    if case.get("classifier"):
        result = case["classifier"]
        referred = result.get("referred", result.get("combined_v4_referred"))
        if isinstance(referred, bool):
            decision = "follow-up flagged" if referred else "follow-up not flagged"
            lines.append("Reconstructed saved screening decision: **" + decision + "**.")
        lines.append("<details><summary>Saved score and rule details</summary>\n\n```json\n"
                     + json.dumps(result, indent=2) + "\n```\n</details>")
        if not case.get("marks"):
            lines.append("No region is colored in this saved display: its referral gate is off.")
        else:
            lines.append("Highlighted leads: " + ", ".join(
                sorted({case['leads'][m['lead']] for m in case['marks']})) + ".")
    if case.get("method_results"):
        for result in case["method_results"]:
            role = result.get("role", result.get("method", "Method")).capitalize()
            if "selected_beat_type" in result:
                beat_type = result["selected_beat_type"]
                names = {"V": "ventricular ectopic beat", "N": "beat annotated as normal"}
                lines.append(role + " selected: **" + names.get(beat_type, beat_type) + "**.")
            if result.get("iou") is not None:
                lines.append(role + " QRS overlap with the expert interval: **"
                             + f"{100 * result['iou']:.1f}%**.")
        lines.append("<details><summary>Saved comparison details</summary>\n\n```json\n"
                     + json.dumps(case["method_results"], indent=2) + "\n```\n</details>")
    details = [notes, case.get("selection_reason", case.get("selection", "")),
               "Source: " + str(case["source"]) + "."]
    lines.append("<details><summary>Annotation and source notes</summary>\n\n"
                 + "\n\n".join(str(line) for line in details if line) + "\n\n</details>")
    display(Markdown("\n\n".join(str(line) for line in lines if line)))


## 2. From screening to a highlight

The screening model asks: **does this ECG need follow-up under the saved referral rule?** Its
ranking score is not a calibrated probability that the person has heart disease. The project
sets its operating point with a referral budget fitted to normal ECGs. These saved development
examples used PTB-XL normal records, not recordings from the university's students.

The explanatory map asks: **which recorded region attracted attention or differs from a reference?**
The existing map uses learned model attention for most records and a beat comparison when its
PVC finding score crosses the saved switch threshold. Saved token contributions show which
regions contributed to that attention head's score;
a difference score shows unusual shape. Neither supplies an expert disease-location annotation.

The following 12-lead examples replay saved results. Colored areas identify the saved method and
lead. Some maps are displayed only when the saved referral rule is positive, so an empty map
can also mean that its referral gate was off. It does not rule out the record's annotated
abnormality. A referral and a localized mark can disagree.
Red marks use the saved normal-beat comparison; blue marks use saved attention contributions.
Amber marks show a forced top candidate whose score is below the usual display cutoff.


### Normal ECG annotation

In [ ]:
case = bundle['cases'][0]
describe_record(case)
plot_record(case)
plot_zoom(case)

### A second normal ECG annotation

In [ ]:
case = bundle['cases'][1]
describe_record(case)
plot_record(case)
plot_zoom(case)

### Premature ventricular beat annotation

In [ ]:
case = bundle['cases'][2]
describe_record(case)
plot_record(case)
plot_zoom(case)

### Complete left bundle branch block annotation

In [ ]:
case = bundle['cases'][3]
describe_record(case)
plot_record(case)
plot_zoom(case)

### Complete right bundle branch block annotation

In [ ]:
case = bundle['cases'][4]
describe_record(case)
plot_record(case)
plot_zoom(case)

### Non-diagnostic T abnormalities annotation

In [ ]:
case = bundle['cases'][5]
describe_record(case)
plot_record(case)
plot_zoom(case)

### Combined anteroseptal and inferolateral infarction-pattern annotations

In [ ]:
case = bundle['cases'][6]
describe_record(case)
plot_record(case)
plot_zoom(case)

### Inferior infarction-pattern annotation

In [ ]:
case = bundle['cases'][7]
describe_record(case)
plot_record(case)
plot_zoom(case)

### Left ventricular hypertrophy annotation: not flagged

In [ ]:
case = bundle['cases'][8]
describe_record(case)
plot_record(case)
plot_zoom(case)

## 3. Finding an unusual beat in a real recording

The candidate finds heartbeats from the waveform, lines them up, and compares each one with
the typical shape of the other beats in the same recording. Each beat is left out of its own
reference. It looks for the largest local difference, and associates that region with its
nearest detected beat. This can identify a premature ventricular beat even when the difference
is in its recovery wave rather than exactly at its sharp spike.

The expert beat labels are used afterward to check the answer. They are not given to the
beat-scoring algorithm. Green bands in these examples orient us around expert-labeled ventricular
beats; they are **not expert QRS onset/offset boundaries**. The earlier and candidate marks show
their selected regions. One example succeeds and one illustrates a remaining failure.


### Finding the unusual beat: a successful example

In [ ]:
case = bundle['cases'][9]
describe_record(case)
plot_record(case)

### Finding the unusual beat: a failure example

In [ ]:
case = bundle['cases'][10]
describe_record(case)
plot_record(case)

## 4. Drawing a more accurate QRS box

The earlier method places a fixed interval around each detected R peak. That is easy to compute,
but the same box can be too short for one beat and too long for another.

The candidate follows local changes in the signal to find the QRS start and end, and combines
the timing information from the available leads. Green is the expert's joint QRS interval;
orange is the earlier fixed box; blue is the candidate box. The expert timing is shared across
the two leads, so these pictures cannot prove that one lead contains the disease.

The first example uses the first complete annotated QRS in a prespecified record. The second
deliberately shows a bad interval from the record with the largest average QRS regression.
Individual pictures explain behavior; the aggregate evaluation below measures improvement.


### Tracing QRS boundaries: a successful example

In [ ]:
case = bundle['cases'][11]
describe_record(case)
truth = [m for m in case['marks'] if m.get('method') == 'expert']
for method in ['fixed', 'hybrid']:
    selected = truth + [m for m in case['marks'] if m.get('method') == method]
    plot_record(case, selected, title=case['title'] + ' — ' + method)

### Tracing QRS boundaries: a failure example

In [ ]:
case = bundle['cases'][12]
describe_record(case)
truth = [m for m in case['marks'] if m.get('method') == 'expert']
for method in ['fixed', 'hybrid']:
    selected = truth + [m for m in case['marks'] if m.get('method') == method]
    plot_record(case, selected, title=case['title'] + ' — ' + method)

### What the completed experiments found

The pictures above explain individual examples. The following numbers come from the full,
previously completed real-data evaluations, not from this small illustrated selection.

| Measurement | Earlier method | Candidate method |
|---|---:|---:|
| Top-scoring beat is ventricular ectopic | 74.1% | 89.6% |
| Normal beat incorrectly flagged | 50.4% | 19.1% |
| QRS interval overlap | 68.2% | 79.4% |
| QRS endpoint within 30 milliseconds | 65.5% | 91.5% |

The beat-selection result averages patients' success in mixed 10-second INCART segments:
30 eligible patients. The false-alarm result for beats annotated as normal uses 31 eligible patients at the
experiments' saved thresholds; lower is better. QRS overlap means intersection divided by union,
averaged across 72 QTDB records. Endpoint accuracy counts the start and end separately; it is
not the percentage of fully correct beats.

The estimated gain in selecting the ventricular ectopic beat was 15.4
percentage points, with a patient-bootstrap 95% interval of
6.6 to 25.8 points.
For QRS overlap, the record-bootstrap gain was
11.2 points, with a 95% interval of
8.9 to
13.6 points.
QTDB does not establish independent patient identities for these 72 records.

### Findings and conclusions

1. Comparing each beat with other beats from the same recording helped identify ventricular
   ectopic beats. It still flagged about 19%
   of beats annotated as normal in that evaluation, so a colored region is a reason to inspect the ECG.
2. Following the waveform instead of drawing a fixed-width box improved QRS timing against
   expert annotations. The experiment evaluated 2,582 complete QRS intervals in the 72-record
   primary comparison. This supports better physiological boundaries.
3. A disease code describes the recording. An unusual beat score ranks differences.
   A QRS boundary identifies timing. These three kinds of evidence answer different questions.
   None of them alone identifies the anatomical cause of a heart problem.
4. The selected conduction, ST/T, infarction-pattern, and high-voltage examples show how different
   labeled ECGs look. Their colored marks are existing screening explanations; they are not
   cardiologist-confirmed disease locations. Missing marks and misleading marks remain possible.
5. The new boundary candidate preserves the existing P-wave boundaries. T-wave localization
   has not shown a reliable general improvement. The first-stage classifier was not retrained
   for this notebook, and these candidate localization methods are separate from its saved decisions.

All waveforms here are original patient recordings. No synthetic ECGs, artificial lesions,
augmentation, training, or new accuracy evaluation were used to make this notebook.


## Sources and reproducibility

Real examples come from [PTB-XL](https://physionet.org/content/ptb-xl/1.0.3/),
[INCART](https://physionet.org/content/incartdb/1.0.0/), and
[QT Database](https://physionet.org/content/qtdb/1.0.0/). The source versions, selected record IDs,
sampling rates, annotation notes, and saved-result receipt hashes are stored in this notebook.
The project reports for Experiments 056 and 059 describe the full protocols and comparison rules.

The embedded bundle stores original waveform samples and saved explanation results. Rerunning
this notebook redraws and explains those results; it does not run the neural networks or repeat
the full benchmarks. No protected final-test cohort is accessed.


In [ ]:
print(json.dumps(bundle['metrics']['receipts'], indent=2))
print('Example IDs:', ', '.join(str(c['id']) for c in bundle['cases']))